Import necessary Libraries

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset
import re
import torch
import numpy as np
from tqdm import tqdm
import requests

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Text pre-processing

In [ ]:
def clean_text(text):
    text = re.sub(r'`.*?`', '', text)
    text = re.sub(r'[\\U00010000-\\U0010ffff]', '', text)
    text = re.sub(r'[\\W]+', ' ', text)
    return text.lower().strip()

GITHUB Details - Replace GITHUB ACCESS KEY with your own Personal access token

In [ ]:
github_token = "GITHUB KEY"
headers = {"Authorization": f"token {github_token}"}

Import the model from Hugging Face

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2).to(device)

Discussion to Issues

Load the dataset

In [ ]:
df = pd.read_csv('../Dataset/DiscussionToIssue.csv')

In [ ]:
encoder = LabelEncoder()
df['IsIssueRaised'] = encoder.fit_transform(df['IsIssueRaised'])
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

Training ...

In [ ]:
df_shuffled['concatenated'] = (df_shuffled['Title'] + ' ' + df_shuffled['Description'] + ' ' + df_shuffled['Comments']).apply(clean_text)
inputs = tokenizer(list(df_shuffled['concatenated']), truncation=True, padding=True, return_tensors="pt", max_length=512)
labels = torch.tensor(df_shuffled['IsIssueRaised'].values)
dataset = TensorDataset(inputs['input_ids'], inputs['attention_mask'], labels)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
model.train()

for epoch in range(3):
    for batch in train_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

Testing ...

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Accuracy:", accuracy_score(all_labels, all_preds))
print("\nClassification Report:\n", classification_report(all_labels, all_preds))

Discussion to Issues with Description Alone

Title + Description

Load the tokenizer and model from Hugging Face

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2).to(device)

Training ...

In [ ]:
df_shuffled['concatenated'] = (df_shuffled['Title'] + ' ' + df_shuffled['Description']).apply(clean_text)
inputs = tokenizer(list(df_shuffled['concatenated']), truncation=True, padding=True, return_tensors="pt", max_length=512)
labels = torch.tensor(df_shuffled['IsIssueRaised'].values)
dataset = TensorDataset(inputs['input_ids'], inputs['attention_mask'], labels)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
model.train()

for epoch in range(3):
    for batch in train_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

Testing ...

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Accuracy:", accuracy_score(all_labels, all_preds))
print("\nClassification Report:\n", classification_report(all_labels, all_preds))

Discussion to Issues with First Comment alone

Title + Description + First Comment

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2).to(device)

Function to extract repository and discussion number

In [ ]:
def extract_github_path(url):
    match = re.search(r'github\.com/([^?#]*)', url)
    return match.group(1) if match else None

Download the Comments

In [ ]:
df_shuffled['Comment'] =  None
for index,row in tqdm(df.iterrows()):
  repo = extract_github_path(row['Issue'])
  curl = f'https://api.github.com/repos/{repo}/comments'
  repo_comment=[]
  cresponse = requests.get(curl,  headers=headers)
  if cresponse.status_code == 200:
    issueComments = cresponse.json()
    cnt=0
    for comment in issueComments:
        repo_comment.append(comment['body'])
        cnt+=1
        if(cnt<1):
          break
  df_shuffled.at[index, 'Comment'] = repo_comment

Training ...

In [ ]:
df_shuffled['concatenated'] = (df_shuffled['Title'] + ' ' + df_shuffled['Description']+' '+str(df_shuffled['Comment'])).apply(clean_text)
inputs = tokenizer(list(df_shuffled['concatenated']), truncation=True, padding=True, return_tensors="pt", max_length=512)
labels = torch.tensor(df_shuffled['IsIssueRaised'].values)
dataset = TensorDataset(inputs['input_ids'], inputs['attention_mask'], labels)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
model.train()

for epoch in range(3):
    for batch in train_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

Testing ...

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Accuracy:", accuracy_score(all_labels, all_preds))
print("\nClassification Report:\n", classification_report(all_labels, all_preds))

Issues to Discussion

Title + Description + Comments

Load the dataset

In [ ]:
df1 = pd.read_csv('../Dataset/IssueToDiscussion.csv')

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2).to(device)

In [ ]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

Training ...

In [ ]:
df1_shuffled['concatenated'] = (df1_shuffled['Title'] + ' ' + df1_shuffled['Description'] + ' ' + df1_shuffled['Comments']).apply(clean_text)
inputs = tokenizer(list(df1_shuffled['concatenated']), truncation=True, padding=True, return_tensors="pt", max_length=512)
labels = torch.tensor(df1_shuffled['ConvertedFromIssue'].values)
dataset = TensorDataset(inputs['input_ids'], inputs['attention_mask'], labels)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
model.train()

for epoch in range(3):
    for batch in train_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

Testing ...

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Accuracy:", accuracy_score(all_labels, all_preds))
print("\nClassification Report:\n", classification_report(all_labels, all_preds))

Issues to Discussion with Description Alone

In [ ]:
df1 = pd.read_csv('../Dataset/IssueToDiscussion.csv')

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2).to(device)

In [ ]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

Training ...

In [ ]:
df1_shuffled['concatenated'] = (df1_shuffled['Description']).apply(clean_text)
inputs = tokenizer(list(df1_shuffled['concatenated']), truncation=True, padding=True, return_tensors="pt", max_length=512)
labels = torch.tensor(df1_shuffled['ConvertedFromIssue'].values)
dataset = TensorDataset(inputs['input_ids'], inputs['attention_mask'], labels)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
model.train()

for epoch in range(3):
    for batch in train_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

Testing ...

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Accuracy:", accuracy_score(all_labels, all_preds))
print("\nClassification Report:\n", classification_report(all_labels, all_preds))

Issues to Discussion with Description and First Comment

In [ ]:
df1 = pd.read_csv('../Dataset/IssueToDiscussion.csv')

In [ ]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

Extract Repository and Issue number

In [ ]:
def extract_github_path(url):
    match = re.search(r'github\.com/([^?#]*)', url)
    return match.group(1) if match else None

Download the Comments

In [ ]:
df1_shuffled['Comment'] =  None
for index,row in tqdm(df1_shuffled.iterrows()):
  repo = extract_github_path(row['Issue'])
  curl = f'https://api.github.com/repos/{repo}/comments'
  repo_comment=[]
  cresponse = requests.get(curl,  headers=headers)
  if cresponse.status_code == 200:
    issueComments = cresponse.json()
    cnt=0
    for comment in issueComments:
        repo_comment.append(comment['body'])
        cnt+=1
        if(cnt<1):
          break
  df1_shuffled.at[index, 'Comment'] = repo_comment

In [ ]:
Training ...

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2).to(device)

In [ ]:
df1_shuffled['concatenated'] = (df1_shuffled['Description'] + ' ' + str(df1_shuffled['Comment'])).apply(clean_text)
inputs = tokenizer(list(df1_shuffled['concatenated']), truncation=True, padding=True, return_tensors="pt", max_length=512)
labels = torch.tensor(df1_shuffled['ConvertedFromIssue'].values)
dataset = TensorDataset(inputs['input_ids'], inputs['attention_mask'], labels)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
model.train()

for epoch in range(3):
    for batch in train_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

Testing ...

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Accuracy:", accuracy_score(all_labels, all_preds))
print("\nClassification Report:\n", classification_report(all_labels, all_preds))

Issues to Discussion with only Comments

In [ ]:
df1 = pd.read_csv('../Dataset/IssueToDiscussion.csv')

Training ...

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2).to(device)

In [ ]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df1_shuffled['concatenated'] = (df1_shuffled['Comments']).apply(clean_text)
inputs = tokenizer(list(df1_shuffled['concatenated']), truncation=True, padding=True, return_tensors="pt", max_length=512)
labels = torch.tensor(df1_shuffled['ConvertedFromIssue'].values)
dataset = TensorDataset(inputs['input_ids'], inputs['attention_mask'], labels)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
model.train()

for epoch in range(3):
    for batch in train_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

Testing ...

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Accuracy:", accuracy_score(all_labels, all_preds))
print("\nClassification Report:\n", classification_report(all_labels, all_preds))